In [ ]:
import os
import pandas as pd

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR

check_and_make_directories([TRAINED_MODEL_DIR])

In [ ]:
#Load Data from 01_data_ops_notebook
train = pd.read_csv('../data/train.csv')
date = train.columns[0]
train.index = pd.factorize(train[date])[0]
train.index.names = ['']
train.head()


In [ ]:
#Setup Environment
stock_dimensions = len(train.tic.unique())
state_space = 1 + 2 * stock_dimensions + len(INDICATORS)*stock_dimensions
print(f"Stock Dimensions:  {stock_dimensions}, State Space: {state_space}")

In [ ]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimensions
num_stock_shares = [0] * stock_dimensions

env_tweeks = {
    "hmax": 100,
    "initial_amount": 1000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimensions,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimensions,
    "reward_scaling": 1e-4
}
#Setup of the environment with given attributes and data set
e_training_gym = StockTradingEnv(df = train, **env_tweeks)


In [ ]:
#Environment for Training

env_train, _ = e_training_gym.get_sb_env()
print(type(env_train))

In [ ]:
#Setup Agent

agent = DRLAgent(env = env_train)

#Set if_using_MODEL = true if u want to use other algorithms
if_using_a2c = False
if_using_ddpg = True
if_using_ppo = False
if_using_td3 = False
if_using_sac = False

In [ ]:
#Using model A2C
model_a2c =agent.get_model("a2c")

if if_using_a2c:
    #setup new logger
    tmp_path = RESULTS_DIR + '/a2c'
    new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_a2c.set_logger(new_logger_a2c)

#Using model DDPG
model_ddpg =agent.get_model("ddpg")

if if_using_ddpg:
    #setup new logger
    tmp_path = RESULTS_DIR + '/ddpg'
    new_logger_ddpg = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_ddpg.set_logger(new_logger_ddpg)

#Using model PPO
model_ppo =agent.get_model("ppo")

if if_using_ppo:
    #setup new logger
    tmp_path = RESULTS_DIR + '/ppo'
    new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_ppo.set_logger(new_logger_ppo)

#Using model TD3
##model_td3 =agent.get_model("td3")

#if if_using_td3:
    #setup new logger
    #tmp_path = RESULTS_DIR + '/td3'
    #new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
   # model_td3.set_logger(new_logger_td3)

#Using model SAC
model_sac =agent.get_model("sac")

if if_using_sac:
    #setup new logger
    tmp_path = RESULTS_DIR + '/sac'
    new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_sac.set_logger(new_logger_sac)

In [ ]:
#Use loggs to train data. Add aditional blocks for other models if needed
if if_using_a2c:
    trained_a2c = agent.train_model(
        model = model_a2c,
        tb_log_name = 'a2c',
        total_timesteps = 500 #Increase for more learning
    )
if if_using_ddpg:
    trained_ddpg = agent.train_model(
        model = model_ddpg,
        tb_log_name = 'ddpg',
        total_timesteps = 500 #Increase for more learning
    )
if if_using_ppo:
    trained_ppo = agent.train_model(
        model = model_ppo,
        tb_log_name = 'ppo',
        total_timesteps = 500 #Increase for more learning
    )


In [ ]:
if if_using_a2c:
    trained_a2c.save(TRAINED_MODEL_DIR + "/agent_a2c")if if_using_a2c else None
if if_using_ddpg:
    trained_ddpg.save(TRAINED_MODEL_DIR + "/agent_ddpg")if if_using_ddpg else None
if if_using_ppo:
    trained_ppo.save(TRAINED_MODEL_DIR + "/agent_ppo")if if_using_ppo else None
